## Inspecting Fields

Wie lässt sich herausfinden, welche Fields eines Models es gibt und welche davon beim Erzeugen ausdrücklich gesetzt wurden? Zwei Attribute beantworten das: `model_fields` beschreibt die Definition und gehört zur Klasse, `model_fields_set` hält fest, was eine einzelne Instanz bekommen hat.

**Danach ist klar:**
- was `model_fields` über ein Model verrät und warum es auf der Klasse gelesen wird,
- was `model_fields_set` enthält und wie sich damit „gesetzt“ von „nur Default“ unterscheiden lässt,
- wie `model_dump(include=...)` nur die gesetzten Fields ausgibt.

Voraussetzung: Notebook `08 - Combining Nullable and Optional`.

In [ ]:
# `BaseModel` definiert das Model, `ValidationError` bleibt für Fehlerfälle importiert.
from pydantic import BaseModel, ValidationError

In [ ]:
# Alle vier Fields haben einen Default und dürfen deshalb beim Erzeugen fehlen.
class Circle(BaseModel):
    center_x: int = 0
    center_y: int = 0
    radius: int = 1
    name: str | None = None

In [ ]:
# `model_fields` wird auf der Klasse gelesen und beschreibt jedes Field mit Annotation und Default.
Circle.model_fields

In [ ]:
# Drei Instanzen, die jeweils andere Fields ausdrücklich setzen.
c1 = Circle(radius=2)
c2 = Circle(name="unit circle")
c3 = Circle(radius=1, name="test")

In [ ]:
# Die Darstellung zeigt alle Fields, auch die mit Default.
c1

In [ ]:
# Hier wurde nur `name` übergeben, die übrigen Werte stammen vom Default.
c2

In [ ]:
# `model_dump()` gibt alle Fields aus, auch die nur mit Default. Ob ein Wert angegeben wurde, verrät es nicht.
c1.model_dump()

**Vorhersage:** `c1` wurde mit `radius=2` erzeugt, `c2` mit `name="unit circle"`, `c3` mit beidem. Was enthält `model_fields_set` jeweils?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** `model_fields_set` ist ein `set` mit den Namen der ausdrücklich gesetzten Fields: `{'radius'}` für `c1`, `{'name'}` für `c2`, beide für `c3`. Fields, die nur ihren Default haben, fehlen.

Kurz gesagt: Das Set zeigt, was beim Erzeugen angegeben wurde, nicht, welchen Wert ein Field hat.
</details>

In [ ]:
c1.model_fields_set

In [ ]:
c2.model_fields_set

In [ ]:
c3.model_fields_set

### Fields, die nur den Default haben

Wie lässt sich umgekehrt ermitteln, welche Fields nicht gesetzt wurden? Die Differenz aus allen Fields der Klasse und dem Set liefert genau diese.

In [ ]:
# `type(c1)` liefert die Klasse, denn `model_fields` über die Instanz zu lesen ist seit Pydantic 2.11 veraltet.
# Die Differenz enthält die Fields, die nur ihren Default haben.
type(c1).model_fields.keys() - c1.model_fields_set

In [ ]:
# Über den Klassennamen ergibt sich dasselbe Ergebnis.
Circle.model_fields.keys() - c1.model_fields_set

In [ ]:
# Bei `c2` bleiben alle Fields außer `name` übrig.
type(c2).model_fields.keys() - c2.model_fields_set

### Ausdrücklich gesetzt oder nur Default?

Zählt ein Field als gesetzt, wenn der übergebene Wert dem Default entspricht? Dafür dient ein zweites Model mit Pflicht-Field und mehreren Defaults.

In [ ]:
# `field_3` ist Pflicht, die übrigen drei Fields haben einen Default.
class Model(BaseModel):
    field_1: int = 1
    field_2: int | None = None
    field_3: str
    field_4: str | None = "Python"

In [ ]:
# `m1` setzt nur das Pflicht-Field. `m2` übergibt alle Fields, die meisten mit dem Default-Wert. `m3` weicht überall vom Default ab.
m1 = Model(field_3="m1")
m2 = Model(field_1=1, field_2=None, field_3="m2", field_4="Python")
m3 = Model(field_1=10, field_2=20, field_3="m3", field_4="Pydantic")

In [ ]:
# Auch hier erscheinen alle vier Fields, ob angegeben oder nicht.
m1.model_dump()

In [ ]:
# Nur das Pflicht-Field `field_3` wurde angegeben.
m1.model_fields_set

In [ ]:
# Mit dem Set als `include` bleiben nur die ausdrücklich gesetzten Fields übrig. Dasselbe leistet `model_dump(exclude_unset=True)`.
m1.model_dump(include=m1.model_fields_set)

**Vorhersage:** `m2` übergibt alle vier Fields, drei davon mit dem Default-Wert. Enthält `model_fields_set` sie trotzdem, und was gibt `include` aus?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Ja, alle vier Fields stehen im Set und in der Ausgabe. Das Set merkt sich nur, ob ein Field angegeben wurde, nicht ob der Wert vom Default abweicht.

Kurz gesagt: „Gesetzt“ bedeutet „angegeben“, nicht „anders als der Default“.
</details>

In [ ]:
m2.model_dump(include=m2.model_fields_set)

### Zusammenfassung

- `model_fields` beschreibt die Definition (Annotation, Default, Pflicht) und wird auf der Klasse gelesen.
- `model_fields_set` gehört zur Instanz und enthält die Namen der Fields, die ausdrücklich angegeben wurden.
- Ein Field mit Default-Wert zählt als gesetzt, sobald es übergeben wurde, auch bei gleichem Wert.
- Die Differenz `Model.model_fields.keys() - instanz.model_fields_set` liefert die Fields, die nur ihren Default haben.
- `model_dump(include=instanz.model_fields_set)` gibt nur die gesetzten Fields aus, wie `model_dump(exclude_unset=True)`.

Weiter mit `10 - JSON Schema Generation`.